# Experiment 4: Exp 3-matched uncertainty-aware MF-MIA against DP targets

This study adds DP training to the main
`exp_3_expanded_missing_feature_baseline_mlp.ipynb` recipe, rather than using
the separate early-stopping or intermediate Exp 3 variants.

**Target configuration:** ACS 2023, nominal PINCP > 50,000, 48 inputs,
100,000 training rows, seed 42, MLP (512, 256, 128), ReLU, dropout 0,
Adam learning rate 0.001, weight decay 0, logical batch size 256 and exactly
100 epochs. Final weights are retained; validation is monitoring only.
This is an overfitting-prone recipe, not a guarantee that DP targets overfit.

**Matched auxiliary attack:** the full eligible ACS 2014 pool, in the original
row order; 20 non-private shadows (6 RF, 5 gradient boosting, 5 XGBoost,
4 MLPs). Keep the original 16-tree sequence, then append four 100-epoch MLPs
matching the target recipe. Use seeds 42 + 10 * index, independent 40/20/40
splits and training-only shadow preprocessing. Query 2,000 IN and 2,000 OUT
records per shadow; retain grouped attack splits, auxiliary roles and thresholds.

**Matched missing-feature study:** the same seven masks, K=6, reconstruction
seed 1042 and forest-leaf donor chains. Compare marginal PFV, conditional
single, conditional average, RF mean-only and matched RF mean+std. The four
conditional controls reuse the same K scores. All components and cutoffs are
fitted/calibrated on auxiliary data once and frozen across target privacy budgets.
The historical Exp 3 blind-walk asymmetry is deliberately retained for this
comparison: JWMNP/JWRIP are categorical in shadows and numeric in targets.
It remains an interpretation limitation, not a correction to the historical run.

**DP conditions:** one fresh non-private control and epsilon = 0.5, 1, 2, 5,
10, 25, 50, constant delta = 1e-6. Private targets use clipping, Gaussian noise,
Poisson sampling and DP-Adam, with accounting calibrated to the full 100 epochs.
Physical batches remain capped at 32; the logical batch size stays 256.

**Necessary difference from historical Exp 3:** fit the shared target preprocessor
on 10,000 public 2014 rows, rather than protected 2023 training rows. Those public
rows are also eligible for shadows/donors, preserving the full Exp 3 auxiliary
pool and its split procedure. Public preprocessing never accesses protected
target records. Its overlap with public auxiliary data is allowed and disclosed;
the reconstruction/meta/calibration/test roles themselves remain disjoint.
The fresh non-private target uses this same fixed preprocessor, so comparisons
within this sweep hold it constant. Historical Exp 3 accuracy (including 62.625%)
is context, not a numerically interchangeable control. DP clipping and Poisson
sampling are additional differences between private and ordinary Adam training.

**Threat model:** a simulated black-box API returns both class probabilities for
native fields and permits repeated/blind-walk queries. The attacker knows the
schema, hidden-field mask, income class for the loss feature, and target recipe;
it does not access target weights, gradients or protected training records.
Target membership labels are evaluation-only; known shadow memberships train
the attacks. No features are hidden during target/shadow training.

Fit auxiliary components once, then train/query/release each target. Preserve
the existing live progress log and atomic per-target final-metric exports.
Internal research checkpoints now save fitted models, per-epoch optimizer/
accountant/RNG states and completion scores for recovery. They are not DP
releases and must remain private. CSV metrics are still saved per target.
Graphs are omitted; plot the exported CSVs separately. This 100-epoch, 20-shadow study is more expensive than the
previous 30-epoch version. Creating/editing it does not authorize execution.

## Dependencies and fixed study configuration
Use the project virtual environment and existing requirements.txt.
All source settings below match the main 100-epoch Exp 3 baseline except the
explicit DP mechanics and public target preprocessing. Private targets use
Opacus per-example clipping, Gaussian noise, Poisson sampling and an RDP
accountant. Physical batch cap 32 and logical batch size 256 are distinct.
Four attack features, including blind walks, remain enabled. Shadow and target
commuting-field walk behavior matches the historical Exp 3 asymmetry.

In [1]:
from pathlib import Path
from importlib.util import find_spec
import csv
import gc
import importlib.metadata
import json
import math

missing_dependencies = [name for name in ['opacus', 'xgboost'] if find_spec(name) is None]
if missing_dependencies:
    raise ImportError('Missing ' + ', '.join(missing_dependencies)
        + '. Install in this notebook kernel: python -m pip install -r requirements.txt')

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.model_selection import train_test_split
from src.data import split_acs_data, subset_target_training, TARGET
import src.expanded_data as expanded_data_module
importlib.reload(expanded_data_module)
from src.expanded_data import EXPANDED_FEATURES, load_expanded_year, reference_split_config
from src.dp_experiment import (
    prepare_public_preprocessing, train_dp_study_target,
)
from src.training import evaluate_model
import src.shadow_models as shadow_models_module
importlib.reload(shadow_models_module)
from src.shadow_models import (train_shadow_model, summarize_shadow_models,
    PAPER_SHADOW_ALGORITHMS)
from src.membership_attack import train_membership_classifier, evaluate_attack
from src.paper_attack_features import collect_shadow_paper_features, collect_target_paper_features
from src.notebook_output import CellProgress, quiet_output, suppress_sklearn_parallel_warning

suppress_sklearn_parallel_warning()
torch.set_num_threads(4)
SEED = 42
TARGET_YEAR = 2023
AUXILIARY_YEAR = 2014
TRAIN_SIZE = 100_000
PUBLIC_PREPROCESSING_SIZE = 10_000
EPSILON_BUDGETS = (0.5, 1, 2, 5, 10, 25, 50)
DELTA = 1e-6
assert 0 < DELTA < 1 / TRAIN_SIZE
assert len(EXPANDED_FEATURES) == 48
EXP3_REFERENCE_TARGET_CONFIG = dict(hidden_dims=(512, 256, 128), dropout=0.0, epochs=100, batch_size=256, learning_rate=0.001, weight_decay=0.0, seed=SEED, device='cpu', early_stopping=False, restore_best=False, monitor_train=True)
N_SHADOW_MODELS = 20
SHADOW_RECORDS_PER_CLASS = 2000
SHADOW_ALGORITHMS = PAPER_SHADOW_ALGORITHMS + ('mlp',)
SHADOW_MLP_CONFIG = {key: value for key, value in EXP3_REFERENCE_TARGET_CONFIG.items() if key != 'seed'}
N_TREE_SHADOWS = 16
SHADOW_PLAN = tuple(PAPER_SHADOW_ALGORITHMS[index % len(PAPER_SHADOW_ALGORITHMS)]
    for index in range(N_TREE_SHADOWS)) + ('mlp',) * 4
assert N_SHADOW_MODELS == len(SHADOW_PLAN) == 20
assert {algorithm: SHADOW_PLAN.count(algorithm) for algorithm in SHADOW_ALGORITHMS} == {
    'random_forest': 6, 'gradient_boosting': 5, 'xgboost': 5, 'mlp': 4}
SPLIT_CONFIG = reference_split_config()
assert SPLIT_CONFIG['seed'] == SEED
TRAIN_CONFIG = dict(
    hidden_dims=EXP3_REFERENCE_TARGET_CONFIG['hidden_dims'],
    dropout=EXP3_REFERENCE_TARGET_CONFIG['dropout'],
    max_epochs=EXP3_REFERENCE_TARGET_CONFIG['epochs'],
    batch_size=EXP3_REFERENCE_TARGET_CONFIG['batch_size'],
    learning_rate=EXP3_REFERENCE_TARGET_CONFIG['learning_rate'],
    weight_decay=EXP3_REFERENCE_TARGET_CONFIG['weight_decay'],
    seed=EXP3_REFERENCE_TARGET_CONFIG['seed'], device=EXP3_REFERENCE_TARGET_CONFIG['device'],
    early_stopping=EXP3_REFERENCE_TARGET_CONFIG['early_stopping'],
    restore_best=EXP3_REFERENCE_TARGET_CONFIG['restore_best'],
    patience=5, min_delta=0.0, max_physical_batch_size=32, max_grad_norm=1.0,
    selection_metric='log_loss', strict_accounting=True)
assert TRAIN_CONFIG['max_epochs'] == 100
assert not TRAIN_CONFIG['early_stopping'] and not TRAIN_CONFIG['restore_best']
# Match the historical Exp 3 protocol exactly: extra commute fields are
# categorical in shadow walks, numeric in target walks. This is a limitation.
SHADOW_NUMERIC_WALK = None
WALK_CONFIG = dict(max_steps=20, repeats=2, categorical_change_probability=0.05,
    batch_size=512, seed=SEED)
EXPANDED_NUMERIC_WALK = {'JWMNP': (2.0, 0, 200), 'JWRIP': (1.0, 1, 10)}
from datetime import datetime, timezone
from os import environ
from tempfile import TemporaryDirectory
import time

output_dir = environ.get('MF_MIA_RUN_DIR')
# This is a parent directory. Compatible code/data/settings select the same
# run-<signature> child on restart; changed inputs select a separate child.
ROOT_RUN_DIR = Path(output_dir) if output_dir else Path(
    'artifacts/exp4_dp_mf_resume')
ROOT_RUN_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR = ROOT_RUN_DIR  # Startup progress only; resolved after data validation.
RUN_STARTED = time.monotonic()


def log_progress(message):
    timestamp = datetime.now(timezone.utc).isoformat(timespec='seconds')
    elapsed = time.monotonic() - RUN_STARTED
    with (RUN_DIR / 'progress.log').open('a', encoding='utf-8') as log:
        print(f'{timestamp} | +{elapsed:.1f}s | {message}', file=log)


class RunProgress(CellProgress):
    def update(self, message):
        log_progress(message)
        super().update(message)


log_progress('Checking ACS input files and recording package versions')

for year, filename in [(TARGET_YEAR, 'psam_p06.csv'), (AUXILIARY_YEAR, 'ss14pca.csv')]:
    raw_path = Path('data/raw/folktables') / str(year) / '1-Year' / filename
    with raw_path.open(newline='', encoding='utf-8-sig') as handle:
        columns = set(next(csv.reader(handle)))
    missing_columns = set(EXPANDED_FEATURES + ['PINCP', 'PWGTP']) - columns
    if missing_columns:
        raise ValueError(f'{raw_path}: missing columns {sorted(missing_columns)}')

versions = {name: importlib.metadata.version(name) for name in
    ['torch', 'opacus', 'numpy', 'pandas', 'scikit-learn', 'xgboost']}
print('Budgets:', EPSILON_BUDGETS, '| Constant delta:', DELTA)
print('Epoch cap:', TRAIN_CONFIG['max_epochs'], '| Patience:', TRAIN_CONFIG['patience'])
print('Complete-feature and missing-feature MIA; frozen auxiliary-only reconstruction and aggregation.')
display(pd.Series(versions, name='package_version'))

from src.paper_attack_features import make_probability_oracle
from src.dp_missing_feature import score_missing_feature_target, uncertainty_contrasts
from src.uncertainty_mf_mia import (
    ConditionalDonorChain, auxiliary_roles, completion_scores, summarize_scores,
    fit_aggregation, evaluate_aggregation,
)

UNCERTAINTY_K = 6
RECONSTRUCTION_SEED = SEED + 1000
MASK_FIELDS = ['MAR', 'WKHP', 'COW', 'SEX']
MISSING_MASKS = {'MAR': ['MAR'], 'WKHP': ['WKHP'], 'MAR+COW': ['MAR', 'COW'], 'COW+SEX': ['COW', 'SEX'], 'MAR+WKHP+SEX': ['MAR', 'WKHP', 'SEX'], 'WKHP+COW+SEX': ['WKHP', 'COW', 'SEX'], 'MAR+WKHP+COW+SEX': ['MAR', 'WKHP', 'COW', 'SEX']}
assert UNCERTAINTY_K == 6 and len(MISSING_MASKS) == 7
UNCERTAINTY_DIR = RUN_DIR / 'uncertainty_mf_mia'
from src.dp_mf_checkpoints import CheckpointStore, run_fingerprint
from src.uncertainty_checkpoints import (
    cached_reconstructor, cached_audit_draws, model_context_signature,
)

Budgets: (0.5, 1, 2, 5, 10, 25, 50) | Constant delta: 1e-06
Epoch cap: 100 | Patience: 5
Complete-feature and missing-feature MIA; frozen auxiliary-only reconstruction and aggregation.


torch           2.15.0+cpu
opacus               1.6.0
numpy                2.5.4
pandas               3.1.1
scikit-learn         1.9.1
xgboost              3.4.1
Name: package_version, dtype: str

## Public preprocessing and fixed target partitions
Use the same stratified ACS 2023 splits, 100,000-row training subset, member
and nonmember IDs as Exp 3. All target variants share one fixed representation,
fitted on 10,000 public ACS 2014 rows only. Unlike historical Exp 3, no private
training row fits target scaling, imputation, indicator choices or vocabulary.

Keep the full original 2014 pool for shadows and reconstruction, without
sorting it before shadow splitting. The public preprocessing sample is a subset
of that pool; overlap among public records does not expose protected target data.
Every shadow still fits its own preprocessor on its training split only.
Reconstruction donors exclude all attack/meta/calibration/test candidate IDs.

The fixed 2023 validation set is outside the protected training set and is
public for this study. It is monitored, but does not select or stop the fixed
100-epoch runs. Test and target MIA labels never choose models or cutoffs.
Privacy is record-level and conditional on the fixed partition/public data.
Public 2014 vocabulary can omit 2023 categories; every target shares this limit.

In [2]:
log_progress('Loading ACS data and preparing target partitions')
target_source = load_expanded_year(TARGET_YEAR, income_base_year=None)
reference_partitions = split_acs_data(target_source, **SPLIT_CONFIG)
target_partitions = subset_target_training(reference_partitions, train_size=TRAIN_SIZE, seed=SEED)
for name in ['validation', 'test', 'mia_members', 'mia_nonmembers']:
    np.testing.assert_array_equal(target_partitions[name].index, reference_partitions[name].index)
assert len(target_partitions['train']) == TRAIN_SIZE
assert len(target_partitions['mia_members']) == len(target_partitions['mia_nonmembers'])

auxiliary_source = load_expanded_year(AUXILIARY_YEAR, income_base_year=None)
_, public_preprocessing = train_test_split(auxiliary_source,
    test_size=PUBLIC_PREPROCESSING_SIZE, stratify=auxiliary_source[TARGET], random_state=SEED)
# Preserve full-pool row order, exactly as in the main Exp 3 notebook.
shadow_auxiliary = auxiliary_source.copy()
public_preprocessing = public_preprocessing.sort_index().copy()
assert set(public_preprocessing.index).issubset(shadow_auxiliary.index)
assert set(auxiliary_source.index).isdisjoint(target_source.index)
target_preprocessor, target_arrays = prepare_public_preprocessing(public_preprocessing, target_partitions)
partition_counts = {name: len(frame) for name, frame in target_partitions.items()}
display(pd.Series({**partition_counts, 'public_2014_preprocessing': len(public_preprocessing),
    'public_2014_shadow_pool': len(shadow_auxiliary)}, name='records'))
print('Encoded input dimensions shared by all targets:', target_arrays['train']['X'].shape[1])
del target_source, reference_partitions, auxiliary_source
gc.collect()
# Namespace checkpoints by real data, protocol source, versions and settings.
# Notebook execution outputs are deliberately excluded from the identity.
protocol_source = json.loads(Path('exp_4_differential_privacy_uncertainty_mf_mia.ipynb').read_text(encoding='utf-8'))
protocol_code = [''.join(cell['source']) for cell in protocol_source['cells'] if cell['cell_type'] == 'code']
run_signature = run_fingerprint(dict(target_training=TRAIN_CONFIG,
    shadow_plan=SHADOW_PLAN, shadow_mlp=SHADOW_MLP_CONFIG, seed=SEED,
    records_per_shadow_class=SHADOW_RECORDS_PER_CLASS, masks=MISSING_MASKS,
    k=UNCERTAINTY_K, reconstruction_seed=RECONSTRUCTION_SEED,
    walk=WALK_CONFIG, shadow_numeric=SHADOW_NUMERIC_WALK,
    target_numeric=EXPANDED_NUMERIC_WALK, budgets=EPSILON_BUDGETS,
    delta=DELTA, features=EXPANDED_FEATURES, versions=versions, notebook_code=protocol_code),
    {**target_partitions, 'auxiliary': shadow_auxiliary, 'public_preprocessing': public_preprocessing},
    [Path('src') / name for name in ['data.py', 'expanded_data.py', 'training.py',
        'shadow_models.py', 'membership_attack.py', 'paper_attack_features.py',
        'missing_feature_pfv.py', 'uncertainty_mf_mia.py', 'uncertainty_checkpoints.py',
        'dp_experiment.py', 'dp_missing_feature.py', 'dp_mf_checkpoints.py']])
RUN_DIR = ROOT_RUN_DIR / f'run-{run_signature[:16]}'
RUN_DIR.mkdir(parents=True, exist_ok=True)
UNCERTAINTY_DIR = RUN_DIR / 'uncertainty_mf_mia'
CHECKPOINTS = CheckpointStore(RUN_DIR / 'ckpt', run_signature)
CHECKPOINTS.cached('run_identity', lambda: dict(signature=run_signature, versions=versions))
log_progress('Resolved checkpoint directory; completed work will be reused')
print('Results and resume directory:', RUN_DIR)

validation                    10000
test                          10000
mia_nonmembers                10000
train                        100000
mia_members                   10000
public_2014_preprocessing     10000
public_2014_shadow_pool      183941
Name: records, dtype: int64

Encoded input dimensions shared by all targets: 2250
Results and resume directory: /home/syedkhaj/MF-MIA-nebula/artifacts/dp-uncertainty/run-3734a3b6b6d89418


## Fit the Exp 3 auxiliary attack once
Use the full 2014 pool and exact 20-shadow plan: original 16 trees (6 RF,
5 GB, 5 XGBoost), followed by 4 MLPs. Preserve model names, seeds, 40/20/40
splits, train-only shadow preprocessing and settings from the main Exp 3.
MLPs use the target's 512/256/128 architecture, 100 epochs, no dropout or weight
decay, and final weights. Their seeds are 202, 212, 222 and 232.

Collect 2,000 IN/2,000 OUT observations per shadow. Fit the same complete-feature
RF classifier and grouped record-ID attack splits. The shadow commuting fields
retain categorical walk replacements, as in Exp 3. Keep fitted models until
held-out auxiliary completion scoring finishes, then release them before targets.
This is a frozen non-private transfer attack, not a DP-budget-adapted attacker.

In [3]:
log_progress('Training and scoring shadow models')
def fit_and_query_shadow(auxiliary, name, algorithm, seed):
    with quiet_output():
        result = train_shadow_model(auxiliary, algorithm, seed=seed, schema='expanded_acs',
            parameters=SHADOW_MLP_CONFIG if algorithm == 'mlp' else None)
        if algorithm == 'mlp':
            assert result['metadata']['epochs_run'] == SHADOW_MLP_CONFIG['epochs']
            assert result['metadata']['returned_epoch'] == SHADOW_MLP_CONFIG['epochs']
            assert result['metadata']['hidden_dims'] == list(SHADOW_MLP_CONFIG['hidden_dims'])
            assert result['metadata']['seed'] == seed
            assert not result['metadata']['early_stopping'] and not result['metadata']['restore_best']
        result['attack_data']['shadow_model'] = name
        features, audits = collect_shadow_paper_features({name: result}, auxiliary,
            per_class=SHADOW_RECORDS_PER_CLASS, config=WALK_CONFIG, seed=SEED,
            include_blind_walk=True, numeric_protocol=SHADOW_NUMERIC_WALK)
    summary = summarize_shadow_models({name: result})
    return features, audits[name], summary, dict(model=result['model'], preprocessor=result['preprocessor'], metadata=result['metadata'], algorithm=algorithm)

shadow_results = {}
shadow_progress = RunProgress()
shadow_feature_parts, shadow_summary_parts, shadow_query_audits, shadow_seeds = [], [], {}, {}
for index, algorithm in enumerate(SHADOW_PLAN):
    name = f'{algorithm}_{index + 1:02d}'
    shadow_seeds[name] = SEED + 10 * index
    shadow_progress.update(f'Shadow {index + 1}/{N_SHADOW_MODELS}: training/querying {name}')
    features, audit, summary, fitted = CHECKPOINTS.cached(f'shadows/{name}',
        lambda: fit_and_query_shadow(shadow_auxiliary, name, algorithm, shadow_seeds[name]))
    shadow_results[name] = fitted
    shadow_feature_parts.append(features)
    shadow_summary_parts.append(summary)
    shadow_query_audits[name] = audit
    del features, audit, summary, fitted
    gc.collect()

shadow_attack_data = pd.concat(shadow_feature_parts, ignore_index=True)
shadow_income_metrics = pd.concat(shadow_summary_parts)
assert len(shadow_attack_data) == N_SHADOW_MODELS * 2 * SHADOW_RECORDS_PER_CLASS
assert shadow_attack_data['shadow_model'].nunique() == N_SHADOW_MODELS
with quiet_output():
    attack_result = CHECKPOINTS.cached('base_attack', lambda: train_membership_classifier(
        shadow_attack_data, seed=SEED, algorithm='random_forest', feature_set='paper'))
split_ids = {name: set(frame['record_id']) for name, frame in attack_result['splits'].items()}
assert split_ids['train'].isdisjoint(split_ids['validation'] | split_ids['test'])
assert split_ids['validation'].isdisjoint(split_ids['test'])
auxiliary_attack_metrics = pd.DataFrame(attack_result['metrics']).T
shadow_progress.update('All shadows scored; auxiliary attack and thresholds frozen')
display(auxiliary_attack_metrics[['accuracy', 'balanced_accuracy', 'roc_auc', 'tpr_at_1pct_fpr_roc']].round(4))
del shadow_feature_parts, shadow_summary_parts
gc.collect()

,accuracy,balanced_accuracy,roc_auc,tpr_at_1pct_fpr_roc
train,0.5722,0.5703,0.6291,0.0410
validation,0.5362,0.5405,0.5596,0.0232
test,0.5355,0.5370,0.5617,0.0223


0

## Auxiliary-only reconstruction and uncertainty calibration
Use the same full public 2014 pool as Exp 3. Public target-preprocessing
rows may overlap this auxiliary pool; the protected 2023 data remain disjoint. Base-attack training IDs never enter the
conditional meta/calibration/test tables. Repeated IDs retain a single role
across shadows. Donors exclude all base-attack/meta/calibration/test IDs.
Half the base validation IDs fit aggregators; the other half calibrate them.
The base auxiliary test IDs remain held out until evaluation.

Seven predeclared masks cover two singles, two pairs, two triples and all four
of MAR, WKHP, COW, SEX. Related visible predictors remain available. Conditional
chains use the same forest-leaf donor procedure as Exp 3, with no income or
membership conditioning. The sampled distribution is approximate, not a calibrated
posterior. Income remains available separately for the base loss attack feature.

In [4]:
log_progress('Assigning auxiliary data roles')
# Set roles once, before any target model is trained or queried.
acs_auxiliary = shadow_auxiliary  # Same public 2014 pool; no second data load.
assert len(shadow_results) == N_SHADOW_MODELS
assert set(shadow_attack_data['shadow_model']) == set(shadow_results)
uncertainty_roles = auxiliary_roles(acs_auxiliary, attack_result, seed=SEED)
assert set().union(*uncertainty_roles.values()) <= set(shadow_auxiliary.index)
# Public preprocessing overlap is disclosed; auxiliary roles remain pairwise disjoint.
assert set(public_preprocessing.index).issubset(shadow_auxiliary.index)
recon_fit_ids, recon_check_ids = train_test_split(
    sorted(uncertainty_roles['reconstruction']), test_size=0.2, random_state=SEED)
recon_training = acs_auxiliary.loc[recon_fit_ids, EXPANDED_FEATURES]
recon_check = acs_auxiliary.loc[recon_check_ids, EXPANDED_FEATURES]
uncertainty_reconstructors = {}
display(pd.Series({name: len(ids) for name, ids in uncertainty_roles.items()}, name='unique_2014_records'))

reconstruction    120846
base               37857
meta                6309
calibration         6310
test               12619
Name: unique_2014_records, dtype: int64

### Fit and audit conditional reconstruction using 2014 only
Held-out reconstruction checks use only reserved 2014 records, not attack candidates.
Numeric coverage is descriptive empirical coverage of the sampled 90% interval.
Categorical sample-match fraction measures draw accuracy; it is not calibrated
probability accuracy. Report these checks even when attack performance is poor.
Related fields remain visible, so reconstruction may be easy through proxies.

In [5]:
log_progress('Fitting and auditing reconstruction models')
# 2. Fit one conditional reconstruction chain per mask; no target queries
from src.notebook_output import CellProgress, quiet_output
reconstruction_audit_rows = []
reconstruction_progress = RunProgress()
for mask_index, (mask_name, hidden) in enumerate(MISSING_MASKS.items(), start=1):
    mask_started = time.monotonic()
    reconstruction_progress.update(f'[{mask_index}/{len(MISSING_MASKS)}] {mask_name}: fitting reconstruction')
    with quiet_output():
        reconstructor = cached_reconstructor(recon_training, hidden, seed=RECONSTRUCTION_SEED,
            directory=CHECKPOINTS.path('reconstruction'))
    uncertainty_reconstructors[mask_name] = reconstructor
    reconstruction_progress.update(f'[{mask_index}/{len(MISSING_MASKS)}] {mask_name}: auditing {UNCERTAINTY_K} completions')
    with quiet_output():
        samples = list(cached_audit_draws(reconstructor, recon_check, k=UNCERTAINTY_K,
            seed=RECONSTRUCTION_SEED, directory=CHECKPOINTS.path('audit') / mask_name))
    for field in hidden:
        observed = recon_check[field].notna().to_numpy()
        if not observed.any():
            continue
        draws = np.column_stack([sample[field].to_numpy()[observed] for sample in samples])
        truth = recon_check[field].to_numpy()[observed]
        row = dict(mask=mask_name, field=field, records=int(observed.sum()))
        if field == 'WKHP':
            lower, upper = np.quantile(draws, [0.05, 0.95], axis=1)
            row.update(mae_of_sample_mean=float(np.abs(draws.mean(axis=1) - truth).mean()),
                sampled_90pct_coverage=float(((truth >= lower) & (truth <= upper)).mean()),
                mean_interval_width=float((upper - lower).mean()))
        else:
            row['sample_match_fraction'] = float((draws == truth[:, None]).mean())
        reconstruction_audit_rows.append(row)
    reconstruction_progress.update(f'[{mask_index}/{len(MISSING_MASKS)}] {mask_name}: complete ({time.monotonic() - mask_started:.1f}s)')
reconstruction_audit = pd.DataFrame(reconstruction_audit_rows)
display(reconstruction_audit.round(4))
# Export this final audit in the final-results cell.

,mask,field,records,sample_match_fraction,mae_of_sample_mean,sampled_90pct_coverage,mean_interval_width
0,MAR,MAR,24170,0.6214,NaN,NaN,NaN
1,WKHP,WKHP,24170,NaN,8.6707,0.7324,22.3474
2,MAR+COW,MAR,24170,0.6212,NaN,NaN,NaN
3,MAR+COW,COW,24170,0.4975,NaN,NaN,NaN
4,COW+SEX,COW,24170,0.4978,NaN,NaN,NaN
5,COW+SEX,SEX,24170,0.5428,NaN,NaN,NaN
6,MAR+WKHP+SEX,MAR,24170,0.6325,NaN,NaN,NaN
7,MAR+WKHP+SEX,WKHP,24170,NaN,8.8158,0.7268,22.6668
8,MAR+WKHP+SEX,SEX,24170,0.5377,NaN,NaN,NaN
9,WKHP+COW+SEX,WKHP,24170,NaN,8.8758,0.7257,23.0591


### Generate held-out auxiliary completion scores
The original 2,000 member/nonmember samples per shadow are reused. Only meta-fit,
calibration and auxiliary-test records are queried here; base-attack training
records are excluded. Repeated IDs retain the same role across all shadows.
Completion never receives membership labels as a conditioning variable.

Use the historical Exp 3 shadow walk (no extra numeric commuting fields).
Target queries later use Exp 3's numeric commuting-field protocol. This
asymmetry is retained explicitly to match the reference, and limits interpretation.
Auxiliary scoring is performed once, not once per privacy budget.

In [6]:
log_progress('Scoring auxiliary completions')
# 3. K scores per held-out shadow observation, separately for every mask
from src.notebook_output import CellProgress, quiet_output
uncertainty_auxiliary_tables = {}
uncertainty_query_audits = {}
auxiliary_progress = RunProgress()
heldout_ids = set().union(*(uncertainty_roles[role] for role in ['meta', 'calibration', 'test']))
for mask_name, reconstructor in uncertainty_reconstructors.items():
    mask_tables = []
    for shadow_name, shadow in shadow_results.items():
        selected = shadow_attack_data.loc[
            (shadow_attack_data['shadow_model'] == shadow_name)
            & shadow_attack_data['record_id'].isin(heldout_ids),
            ['record_id', 'membership']].sort_values('record_id').reset_index(drop=True)
        if selected.empty:
            continue
        assert selected['record_id'].is_unique
        candidates = acs_auxiliary.loc[selected['record_id']]
        oracle = make_probability_oracle(shadow['model'], shadow['preprocessor'], schema='acs')
        auxiliary_progress.update(f'Auxiliary scoring: {mask_name} / {shadow_name}, {len(selected):,} records, K={UNCERTAINTY_K}')
        with quiet_output():
            draw_scores, audit = completion_scores(oracle, candidates, reconstructor,
                attack_result['classifier'], k=UNCERTAINTY_K, seed=RECONSTRUCTION_SEED,
                walk_config=WALK_CONFIG, include_blind_walk=True, numeric_protocol=SHADOW_NUMERIC_WALK,
                checkpoint_dir=CHECKPOINTS.path('auxiliary_scores') / mask_name / shadow_name,
                context_signature=model_context_signature(shadow['model'], shadow['preprocessor']))
        table = summarize_scores(draw_scores)
        table['record_id'] = selected['record_id'].to_numpy()
        table['membership'] = selected['membership'].to_numpy()
        table['shadow_model'] = shadow_name
        table['role'] = [next(role for role in ['meta', 'calibration', 'test']
            if record_id in uncertainty_roles[role]) for record_id in table['record_id']]
        mask_tables.append(table)
        uncertainty_query_audits[f'{mask_name}/{shadow_name}'] = audit
    uncertainty_auxiliary_tables[mask_name] = pd.concat(mask_tables, ignore_index=True)
auxiliary_progress.update(f'Auxiliary scoring complete: {len(uncertainty_auxiliary_tables)} masks')

In [7]:
log_progress('Fitting and calibrating auxiliary aggregators')
# 4. Fit aggregators, freeze thresholds on calibration, evaluate auxiliary test
from src.notebook_output import quiet_output
uncertainty_aggregators = {}
uncertainty_auxiliary_result_rows = []
for mask_name, table in uncertainty_auxiliary_tables.items():
    log_progress(f'Fitting and calibrating aggregators: {mask_name}')
    meta = table.loc[table['role'] == 'meta']
    calibration = table.loc[table['role'] == 'calibration']
    auxiliary_test = table.loc[table['role'] == 'test']
    with quiet_output():
        learned, thresholds = CHECKPOINTS.cached(f'aggregators/{mask_name}',
            lambda: fit_aggregation(meta, calibration, seed=SEED))
    uncertainty_aggregators[mask_name] = (learned, thresholds)
    with quiet_output():
        metrics, predictions = evaluate_aggregation(auxiliary_test, learned, thresholds)
    uncertainty_auxiliary_result_rows.extend(metrics.assign(mask=mask_name,
        n_missing=len(MISSING_MASKS[mask_name]), dataset='2014_auxiliary_test').to_dict('records'))
print('Auxiliary test: macro averages across masks of the same size')
display(pd.DataFrame(uncertainty_auxiliary_result_rows).groupby(['n_missing', 'method'])[
    ['roc_auc', 'balanced_accuracy', 'tpr_at_1pct_fpr_roc']].mean().round(4))

Auxiliary test: macro averages across masks of the same size


roc_auc  balanced_accuracy  tpr_at_1pct_fpr_roc
n_missing method                                                       
1         average        0.5562             0.5309               0.0214
          learned_mean   0.5396             0.5242               0.0185
          single         0.5511             0.5293               0.0215
          uncertainty    0.5443             0.5262               0.0209
2         average        0.5579             0.5357               0.0246
          learned_mean   0.5418             0.5265               0.0178
          single         0.5496             0.5300               0.0201
          uncertainty    0.5452             0.5261               0.0168
3         average        0.5414             0.5276               0.0204
          learned_mean   0.5341             0.5225               0.0154
          single         0.5388             0.5204               0.0158
          uncertainty    0.5368             0.5193               0.0155
4         average        0.5412             0.5281               0.0169
          learned_mean   0.5382             0.5221               0.0163
          single         0.5292             0.5167               0.0153
          uncertainty    0.5378             0.5225               0.0182

### Freeze the attack and release shadow models
Single and average score cutoffs are selected on conditional calibration scores.
The two matched RFs (300 trees, depth 12, minimum leaf 20) are fitted on the
meta group, using mean alone or mean+std, then calibrated separately. PFV and
complete-feature MIA retain the base attack's original validation thresholds.
Each cutoff stays fixed across all target privacy budgets. Target membership
and test results never select methods, masks, thresholds, epochs, epsilon or K.

In [8]:
log_progress('Freezing auxiliary models and thresholds')
uncertainty_auxiliary_results = pd.DataFrame(uncertainty_auxiliary_result_rows)
assert len(uncertainty_aggregators) == len(MISSING_MASKS)
assert set(uncertainty_aggregators) == set(uncertainty_reconstructors) == set(MISSING_MASKS)
frozen_thresholds = {mask: thresholds for mask, (_, thresholds) in uncertainty_aggregators.items()}
# Keep only public reconstruction/aggregation models and the base classifier.
# No DP target has been trained yet. No retraining of shadows will be needed.
shadow_mlp_training = {name: result['metadata'] for name, result in shadow_results.items()
    if result['algorithm'] == 'mlp'}
del shadow_results, shadow, oracle, uncertainty_auxiliary_tables
del recon_training, recon_check, samples, draws, draw_scores, table, meta, calibration, auxiliary_test
gc.collect()
print('Auxiliary reconstruction, aggregators and thresholds frozen for every target condition.')

Auxiliary reconstruction, aggregators and thresholds frozen for every target condition.


## Train and evaluate one fixed-schedule target at a time
Each fresh target uses the main Exp 3 architecture and unregularized optimizer
recipe for exactly 100 epochs, retaining final weights. Validation loss and
accuracy are monitoring diagnostics only. best_epoch is diagnostic;
returned_epoch must be 100. Private training metrics, test and MIA results do
not select epochs. End-of-epoch private training metrics are not used to stop
or select private models; final train/test utility is evaluation only.

For each private run, calibrate noise to the full 100-epoch logical-step count
and conservative Opacus accountant rate before the first update. Record actual
Poisson sampling rate separately. Fixed-schedule epsilon and final accountant
epsilon must agree and stay within the requested budget. Delta is 1e-6.
Independent nondeterministic DP sampling/noise remain research settings.

Before releasing each model, evaluate complete-feature MIA and all five
missing-feature controls on the same original target IDs, using the frozen
auxiliary pipeline. Complete/PFV/conditional target walks use the original
Exp 3 numeric commuting fields. The same completion seeds and K draws are reused
across budgets. Source target preprocessing is the documented exception;
the fresh non-private control shares that representation with every DP target.

In [9]:
target_progress = RunProgress()

def run_target_condition(epsilon):
    condition = 'non_private' if epsilon is None else f'epsilon_{epsilon:g}'
    condition_started = time.monotonic()
    target_progress.update(f'{condition}: training from scratch')
    with quiet_output():
        model, history, metadata = CHECKPOINTS.cached(f'targets/{condition}/trained',
            lambda: train_dp_study_target(
                target_arrays['train']['X'], target_arrays['train']['y'],
                target_arrays['validation']['X'], target_arrays['validation']['y'],
                epsilon=epsilon, delta=DELTA, progress=target_progress.update,
                checkpoint_path=CHECKPOINTS.path(f'targets/{condition}/training_epoch.joblib'),
                checkpoint_signature=CHECKPOINTS.signature, **TRAIN_CONFIG))
    fit_seconds = time.monotonic() - condition_started
    log_progress(f'{condition}: training finished in {fit_seconds:.1f}s including setup')
    target_progress.update(f'{condition}: evaluating final epoch {metadata["returned_epoch"]}')
    assert metadata['epochs_run'] == metadata['returned_epoch'] == TRAIN_CONFIG['max_epochs'] == 100
    assert not metadata['early_stopping'] and not metadata['restore_best']
    if epsilon is not None:
        np.testing.assert_allclose(metadata['epsilon_fixed_schedule_bound'],
            metadata['epsilon_at_stop_diagnostic'], rtol=0, atol=1e-6)
    income_metrics = {name: evaluate_model(model, target_arrays[name]['X'], target_arrays[name]['y'])
        for name in ['train', 'validation', 'test']}
    target_progress.update(f'{condition}: querying complete-feature membership attack')
    def complete_attack():
        with quiet_output():
            features, audit = collect_target_paper_features(model, target_preprocessor,
                target_partitions, config=WALK_CONFIG, seed=SEED,
                include_blind_walk=True, numeric_protocol=EXPANDED_NUMERIC_WALK)
            metrics, _ = evaluate_attack(attack_result['classifier'], features,
                threshold=attack_result['accuracy_threshold'],
                low_fpr_threshold=attack_result['low_fpr_threshold'])
        return metrics, audit
    mia_metrics, query_audit = CHECKPOINTS.cached(f'targets/{condition}/complete_attack', complete_attack)
    assert mia_metrics['records'] == 2 * mia_metrics['members']
    row = dict(condition=condition, private=epsilon is not None,
        epsilon_requested=epsilon, delta=DELTA if epsilon is not None else None,
        epsilon_fixed_schedule_bound=metadata['epsilon_fixed_schedule_bound'],
        epsilon_at_stop_diagnostic=metadata['epsilon_at_stop_diagnostic'],
        noise_multiplier=metadata['noise_multiplier'],
        sampler_sample_rate=metadata['sampler_sample_rate'],
        accountant_sample_rate=metadata['accountant_sample_rate'],
        bound_sample_rate=metadata['bound_sample_rate'],
        selection_metric=metadata['selection_metric'], epochs_run=metadata['epochs_run'],
        best_epoch=metadata['best_epoch'], returned_epoch=metadata['returned_epoch'],
        training_seconds=metadata['training_seconds'],
        target_train_accuracy=income_metrics['train']['accuracy'],
        target_validation_accuracy=income_metrics['validation']['accuracy'],
        target_test_accuracy=income_metrics['test']['accuracy'],
        target_test_roc_auc=income_metrics['test']['roc_auc'],
        target_test_log_loss=income_metrics['test']['log_loss'],
        generalization_gap=income_metrics['train']['accuracy'] - income_metrics['test']['accuracy'],
        mia_accuracy=mia_metrics['accuracy'], mia_balanced_accuracy=mia_metrics['balanced_accuracy'],
        mia_roc_auc=mia_metrics['roc_auc'], mia_tpr_at_1pct_fpr_roc=mia_metrics['tpr_at_1pct_fpr_roc'],
        mia_tpr_at_shadow_threshold=mia_metrics['tpr_at_shadow_threshold'],
        mia_fpr_at_shadow_threshold=mia_metrics['fpr_at_shadow_threshold'])
    target_progress.update(f'{condition}: scoring seven missing-feature masks')
    query = make_probability_oracle(model, target_preprocessor, schema='acs')
    with quiet_output():
        missing_rows, missing_audits = score_missing_feature_target(
            query, {name: target_partitions[name] for name in ['mia_members', 'mia_nonmembers']},
            shadow_auxiliary, MISSING_MASKS, uncertainty_reconstructors, uncertainty_aggregators,
            attack_result, k=UNCERTAINTY_K, reconstruction_seed=RECONSTRUCTION_SEED,
            pfv_seed=SEED, walk_config=WALK_CONFIG, numeric_protocol=EXPANDED_NUMERIC_WALK,
            progress=lambda message: target_progress.update(f'{condition}: {message}'),
            checkpoint_dir=CHECKPOINTS.path(f'targets/{condition}/missing_features'),
            context_signature=model_context_signature(model, target_preprocessor))
    complete_row = dict(mask='complete', n_missing=0, method='complete', **mia_metrics)
    attack_rows = [complete_row] + missing_rows
    for attack_row in attack_rows:
        attack_row.update(condition=condition, private=epsilon is not None,
            epsilon_requested=epsilon, delta=DELTA if epsilon is not None else None,
            epsilon_fixed_schedule_bound=metadata['epsilon_fixed_schedule_bound'])
    metadata.update(income_metrics=income_metrics, membership_metrics=mia_metrics,
        query_audit=query_audit, missing_feature_query_audits=missing_audits)
    condition_seconds = time.monotonic() - condition_started
    row.update(fit_seconds=fit_seconds, condition_seconds=condition_seconds)
    metadata.update(fit_seconds=fit_seconds, condition_seconds=condition_seconds)
    # Return metrics only: no fitted target, private row scores or model files.
    return row, history, metadata, attack_rows

## Run or resume the eight target conditions
This cell starts/resumes experiments and requires explicit approval. Run the
notebook in order after restarting; identical settings/data/code resolve to the
same checkpoint directory. Completed shadow/reconstruction/attack work is loaded.
Completed target conditions are skipped and their metrics restored. An interrupted
target resumes after its last complete epoch, with optimizer/accountant/RNG restored.
After training, completed masks and conditional draws are reused. An unfinished
epoch or draw is repeated; there are no mid-batch training checkpoints.

Changing code, data, package versions or settings selects a separate run directory
instead of silently reusing incompatible checkpoints. MF_MIA_RUN_DIR optionally
sets the parent directory. Copy the whole compatible run folder to retain resume
state. Do not start concurrent writers to the same run directory.

In [10]:
log_progress('Running target conditions')
def save_target_results(condition, row, history, attack_rows):
    # Publish the whole condition together so an interrupted write is never
    # mistaken for a completed condition.
    targets_dir = RUN_DIR / 'targets'
    targets_dir.mkdir(exist_ok=True)
    with TemporaryDirectory(prefix='.pending-', dir=targets_dir) as temp_dir:
        bundle = Path(temp_dir) / 'results'
        bundle.mkdir()
        pd.DataFrame([row]).to_csv(bundle / 'dp_target_and_mia_metrics.csv', index=False)
        attacks = pd.DataFrame(attack_rows)
        attacks.to_csv(bundle / 'dp_missing_feature_metrics.csv', index=False)
        uncertainty_contrasts(attacks).to_csv(bundle / 'uncertainty_vs_mean_only.csv', index=False)
        pd.DataFrame(history).to_csv(bundle / 'public_validation_history.csv', index=False)
        destination = targets_dir / condition
        if destination.exists():
            # A completed checkpoint may be replayed to rebuild combined tables.
            # Preserve already published matching CSV bundles without renaming over them.
            for expected in bundle.iterdir():
                saved = destination / expected.name
                if not saved.exists() or saved.read_bytes() != expected.read_bytes():
                    raise ValueError(f'Existing results differ from the verified checkpoint: {saved}')
        else:
            bundle.rename(destination)


# Reload presentation support only; keep existing data, shadows and attack.
import importlib
import src.notebook_output as notebook_output_module
importlib.reload(notebook_output_module)
from src.notebook_output import compact_dp_warnings

study_rows, training_histories, target_metadata, missing_feature_rows = [], {}, {}, []
with compact_dp_warnings():
    for epsilon in (None,) + EPSILON_BUDGETS:
        condition_key = 'non_private' if epsilon is None else f'epsilon_{epsilon:g}'
        row, history, metadata, attack_rows = CHECKPOINTS.cached(f'targets/{condition_key}/completed',
            lambda: run_target_condition(epsilon))
        condition = row["condition"]
        save_target_results(condition, row, history, attack_rows)
        target_progress.update(f'{condition}: results saved after {row["condition_seconds"]:.1f}s')
        missing_feature_rows.extend(attack_rows)
        study_rows.append(row)
        training_histories[row['condition']] = history
        target_metadata[row['condition']] = metadata
        gc.collect()
study_results = pd.DataFrame(study_rows)
assert len(study_results) == 1 + len(EPSILON_BUDGETS)
assert study_results.loc[study_results['private'], 'delta'].eq(DELTA).all()
target_progress.update('All eight conditions completed; final metrics ready')

missing_feature_results = pd.DataFrame(missing_feature_rows)
assert len(missing_feature_results) == len(study_results) * (1 + 5 * len(MISSING_MASKS))
assert not missing_feature_results.duplicated(['condition', 'mask', 'method']).any()
assert missing_feature_results['records'].eq(2 * len(target_partitions['mia_members'])).all()
assert missing_feature_results.loc[missing_feature_results['private'], 'delta'].eq(DELTA).all()
uncertainty_changes = uncertainty_contrasts(missing_feature_results)

/home/syedkhaj/MF-MIA-nebula/.venv/lib/python3.12/site-packages/opacus/privacy_engine.py:98: UserWarning: Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retrain one last time before production with ``secure_mode`` turned on.
  warnings.warn(
/home/syedkhaj/MF-MIA-nebula/.venv/lib/python3.12/site-packages/opacus/accountants/analysis/rdp.py:332: UserWarning: Optimal order is the largest alpha. Please consider expanding the range of alphas to get a tighter privacy bound.
  warnings.warn(
/home/syedkhaj/MF-MIA-nebula/src/dp_experiment.py:258: UserWarning: Full backward hook is firing when gradients are computed with respect to module outputs since no inputs require gradients. See https://docs.pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.register_full_backward_hook for more details.
  loss.backward()


## Compare utility, generalization and membership leakage
Accuracies below are fractions; graphs show percentages. The generalization gap
is training accuracy minus test accuracy. A small gap with poor test accuracy can
indicate underfitting, so both are reported. Ordinary MIA accuracy uses one
auxiliary-validation-selected threshold, frozen across targets. ROC-AUC and
TPR@1%FPR supplement accuracy so threshold transfer is visible.

For missing features, inspect every mask and method in the detailed CSV.
Mean+std minus mean-only is the paired descriptive contrast for uncertainty.
Mean results over masks summarize the selected study; masks are not independent
replicates, and those means are not a statistical significance test.
TPR@1%FPR derived from each target ROC is a retrospective evaluation statistic.
Also report TPR and realized FPR at the auxiliary-selected cutoff, which can
differ substantially from 1% after distribution shift or DP training.

In [11]:
display(study_results.set_index('condition')[[
    'epsilon_requested', 'epsilon_fixed_schedule_bound', 'epsilon_at_stop_diagnostic',
    'delta', 'epochs_run', 'returned_epoch', 'best_epoch', 'target_train_accuracy', 'target_test_accuracy',
    'generalization_gap', 'mia_accuracy', 'mia_roc_auc', 'mia_tpr_at_1pct_fpr_roc'
]].round(6))

display(study_results.loc[study_results['private'], [
    'condition', 'sampler_sample_rate', 'accountant_sample_rate', 'bound_sample_rate',
    'noise_multiplier', 'epsilon_fixed_schedule_bound'
]].set_index('condition'))

print('Missing-feature target metrics: descriptive macro averages over the seven masks')
display(missing_feature_results.loc[missing_feature_results['method'] != 'complete'].groupby(
    ['condition', 'method'])[['accuracy', 'balanced_accuracy', 'roc_auc',
        'tpr_at_1pct_fpr_roc', 'tpr_at_shadow_threshold', 'fpr_at_shadow_threshold']].mean().round(4))
print('Per-mask uncertainty minus learned-mean contrasts (positive favors uncertainty):')
display(uncertainty_changes.set_index(['condition', 'mask'])[
    ['accuracy_change', 'roc_auc_change', 'tpr_at_1pct_fpr_roc_change']].round(4))

,epsilon_requested,epsilon_fixed_schedule_bound,epsilon_at_stop_diagnostic,delta,epochs_run,returned_epoch,best_epoch,target_train_accuracy,target_test_accuracy,generalization_gap,mia_accuracy,mia_roc_auc,mia_tpr_at_1pct_fpr_roc
condition,,,,,,,,,,,,,
non_private,NaN,NaN,NaN,NaN,100,100,2,0.99841,0.7854,0.21301,0.63115,0.689239,0.0311
epsilon_0.5,0.5,0.491537,0.491537,0.000001,100,100,1,0.79178,0.7846,0.00718,0.50135,0.504853,0.0109
epsilon_1,1.0,0.993826,0.993826,0.000001,100,100,1,0.80248,0.7961,0.00638,0.50220,0.502487,0.0103
epsilon_2,2.0,1.995987,1.995987,0.000001,100,100,1,0.80959,0.8039,0.00569,0.49835,0.505358,0.0085
epsilon_5,5.0,4.993633,4.993633,0.000001,100,100,1,0.81728,0.8076,0.00968,0.50105,0.502279,0.0088
epsilon_10,10.0,9.991104,9.991104,0.000001,100,100,1,0.82163,0.8075,0.01413,0.50390,0.511248,0.0114
epsilon_25,25.0,24.997075,24.997075,0.000001,100,100,2,0.82570,0.8108,0.01490,0.50360,0.510905,0.0128
epsilon_50,50.0,49.992751,49.992751,0.000001,100,100,1,0.82909,0.8159,0.01319,0.49555,0.504017,0.0087


,sampler_sample_rate,accountant_sample_rate,bound_sample_rate,noise_multiplier,epsilon_fixed_schedule_bound
condition,,,,,
epsilon_0.5,0.002558,0.002564,0.002564,4.531250,0.491537
epsilon_1,0.002558,0.002564,0.002564,2.421875,0.993826
epsilon_2,0.002558,0.002564,0.002564,1.396484,1.995987
epsilon_5,0.002558,0.002564,0.002564,0.827026,4.993633
epsilon_10,0.002558,0.002564,0.002564,0.646973,9.991104
epsilon_25,0.002558,0.002564,0.002564,0.505562,24.997075
epsilon_50,0.002558,0.002564,0.002564,0.428677,49.992751


Missing-feature target metrics: descriptive macro averages over the seven masks


accuracy  balanced_accuracy  roc_auc  \
condition   method                                               
epsilon_0.5 average         0.5053             0.5053   0.5067   
            learned_mean    0.5045             0.5045   0.5061   
            pfv             0.5036             0.5036   0.5058   
            single          0.5035             0.5035   0.5050   
            uncertainty     0.5042             0.5042   0.5059   
epsilon_1   average         0.5030             0.5030   0.5037   
            learned_mean    0.5019             0.5019   0.5037   
            pfv             0.5028             0.5028   0.5065   
            single          0.5025             0.5025   0.5039   
            uncertainty     0.5023             0.5023   0.5046   
epsilon_10  average         0.5003             0.5003   0.5063   
            learned_mean    0.5015             0.5015   0.5055   
            pfv             0.5016             0.5016   0.5091   
            single          0.5012             0.5012   0.5058   
            uncertainty     0.5035             0.5035   0.5066   
epsilon_2   average         0.5007             0.5007   0.5069   
            learned_mean    0.5021             0.5021   0.5063   
            pfv             0.4993             0.4993   0.5050   
            single          0.5024             0.5024   0.5068   
            uncertainty     0.5020             0.5020   0.5057   
epsilon_25  average         0.5010             0.5010   0.5094   
            learned_mean    0.5024             0.5024   0.5084   
            pfv             0.5026             0.5026   0.5112   
            single          0.5021             0.5021   0.5077   
            uncertainty     0.5032             0.5032   0.5080   
epsilon_5   average         0.4994             0.4994   0.5059   
            learned_mean    0.5004             0.5004   0.5047   
            pfv             0.4997             0.4997   0.5032   
            single          0.5022             0.5022   0.5061   
            uncertainty     0.5005             0.5005   0.5037   
epsilon_50  average         0.4995             0.4995   0.5052   
            learned_mean    0.5003             0.5003   0.5056   
            pfv             0.4987             0.4987   0.5060   
            single          0.4999             0.4999   0.5066   
            uncertainty     0.5007             0.5007   0.5058   
non_private average         0.6098             0.6098   0.6498   
            learned_mean    0.6028             0.6028   0.6363   
            pfv             0.5918             0.5918   0.6327   
            single          0.5957             0.5957   0.6349   
            uncertainty     0.6007             0.6007   0.6401   

                          tpr_at_1pct_fpr_roc  tpr_at_shadow_threshold  \
condition   method                                                       
epsilon_0.5 average                    0.0094                   0.0068   
            learned_mean               0.0115                   0.0211   
            pfv                        0.0111                   0.0062   
            single                     0.0102                   0.0071   
            uncertainty                0.0106                   0.0158   
epsilon_1   average                    0.0098                   0.0020   
            learned_mean               0.0108                   0.0099   
            pfv                        0.0109                   0.0028   
            single                     0.0100                   0.0033   
            uncertainty                0.0109                   0.0069   
epsilon_10  average                    0.0094                   0.0000   
            learned_mean               0.0105                   0.0006   
            pfv                        0.0096                   0.0000   
            single                     0.0101                   0.0000   
            uncertainty                0.0109                   0.000

Per-mask uncertainty minus learned-mean contrasts (positive favors uncertainty):


accuracy_change  roc_auc_change  \
condition   mask                                                
epsilon_0.5 COW+SEX                   -0.0003          0.0015   
            MAR                       -0.0005          0.0005   
            MAR+COW                    0.0019         -0.0009   
            MAR+WKHP+COW+SEX           0.0016          0.0003   
            MAR+WKHP+SEX              -0.0080         -0.0034   
            WKHP                      -0.0004          0.0021   
            WKHP+COW+SEX               0.0033         -0.0019   
epsilon_1   COW+SEX                    0.0040          0.0037   
            MAR                       -0.0012         -0.0015   
            MAR+COW                   -0.0008         -0.0022   
            MAR+WKHP+COW+SEX           0.0000          0.0016   
            MAR+WKHP+SEX              -0.0004          0.0000   
            WKHP                      -0.0022         -0.0005   
            WKHP+COW+SEX               0.0033          0.0046   
epsilon_10  COW+SEX                    0.0033          0.0043   
            MAR                        0.0040          0.0039   
            MAR+COW                    0.0033         -0.0005   
            MAR+WKHP+COW+SEX           0.0021          0.0019   
            MAR+WKHP+SEX               0.0029          0.0000   
            WKHP                       0.0045          0.0005   
            WKHP+COW+SEX              -0.0061         -0.0020   
epsilon_2   COW+SEX                    0.0009         -0.0009   
            MAR                        0.0008          0.0014   
            MAR+COW                   -0.0052         -0.0037   
            MAR+WKHP+COW+SEX          -0.0038         -0.0025   
            MAR+WKHP+SEX               0.0019          0.0005   
            WKHP                       0.0044          0.0011   
            WKHP+COW+SEX               0.0004         -0.0003   
epsilon_25  COW+SEX                    0.0050          0.0027   
            MAR                        0.0038          0.0006   
            MAR+COW                   -0.0028         -0.0003   
            MAR+WKHP+COW+SEX          -0.0019         -0.0028   
            MAR+WKHP+SEX               0.0011         -0.0017   
            WKHP                       0.0028          0.0002   
            WKHP+COW+SEX              -0.0020         -0.0018   
epsilon_5   COW+SEX                   -0.0017         -0.0009   
            MAR                        0.0003          0.0006   
            MAR+COW                    0.0023         -0.0007   
            MAR+WKHP+COW+SEX          -0.0005         -0.0002   
            MAR+WKHP+SEX              -0.0009         -0.0044   
            WKHP                       0.0012         -0.0022   
            WKHP+COW+SEX              -0.0003          0.0005   
epsilon_50  COW+SEX                    0.0023          0.0016   
            MAR                       -0.0026         -0.0029   
            MAR+COW                   -0.0002         -0.0004   
            MAR+WKHP+COW+SEX           0.0016         -0.0020   
            MAR+WKHP+SEX              -0.0030          0.0005   
            WKHP                       0.0034          0.0018   
            WKHP+COW+SEX               0.0013          0.0033   
non_private COW+SEX                   -0.0017          0.0031   
            MAR                        0.0018         -0.0003   
            MAR+COW                    0.0026          0.0029   
            MAR+WKHP+COW+SEX          -0.0062          0.0080   
            MAR+WKHP+SEX               0.0047          0.0069   
            WKHP                      -0.0001          0.0016   
            WKHP+COW+SEX              -0.0161          0.0042   

                              tpr_at_1pct_fpr_roc_change  
condition   mask                                          
epsilon_0.5 COW+SEX                              -0.0005  
            MAR                                  -0.0002  
            MAR+COW    

## Save final research results and interpretation limits
Save final CSV metrics, training histories and protocol; plotting is separate.
Research outputs are **not DP-sanitized releases**. Internal checkpoints
also contain fitted models, optimizer/accountant/RNG states and per-record
completion scores. Keep them private and out of Git. Publishing random-
generator states or optimizer checkpoints is not covered by the final-model
DP guarantee. Checkpoints are local recovery files, not report artifacts.

The stated epsilon/delta bound concerns each individual target inference model
with its fixed public preprocessor. Releasing several models trained on the same
records requires privacy composition; the non-private control has no DP
guarantee. `secure_mode=False` is a research setting. All 100 epochs run;
validation does not select weights or stop training. The full-schedule
accountant bound is checked against the final expenditure.

This study has one initialization seed per condition and independently sampled
DP noise. It uses transferred mixed tree/MLP-shadow attacks, native year-specific Census
codes, an auxiliary-derived vocabulary, record-level protection, and a fixed
research partition. Repeated runs and attacks matched to DP targets are follow-up
checks. Lower measured MIA does not by itself demonstrate stronger protection.

**Comparison limits.** Historical Exp 3 fitted target preprocessing on 2023
training data; this DP study uses a fixed public 2014 representation. Compare
against the fresh non-private control rather than interpreting historical
Exp 3 numbers as a matched result. The original shadow/target blind-walk
asymmetry is deliberately preserved and remains a limitation.

**Combined-study limitations.** A frozen non-private, cross-year shadow attack
can be weak against DP targets. Near-chance results do not prove that no stronger
attacker exists. Conditional completion is approximate, visible proxy fields
remain, and K=6 gives noisy std estimates. One seed and one independent DP noise
realization per budget provide exploratory evidence, not reliable repeatability
or optimal training settings. Uncertainty need not outperform mean-only, and
empirical curves need not be monotonic in epsilon.

Repeated queries and auxiliary-only completion/aggregation are post-processing
of each fixed DP inference model and do not themselves spend another training
privacy budget. This does not make the full sweep or its private-data research
metrics a DP release: per-model bounds, composition and the output limitations
above still apply. No attack query or target membership result influences training.

In [12]:
log_progress('Exporting final results')
def json_value(value):
    if isinstance(value, dict):
        return {str(key): json_value(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_value(item) for item in value]
    if isinstance(value, np.generic):
        return json_value(value.item())
    if isinstance(value, float) and not math.isfinite(value):
        return str(value)
    return value

RUN_DIR.mkdir(parents=True, exist_ok=True)
study_results.to_csv(RUN_DIR / 'dp_target_and_mia_metrics.csv', index=False)
UNCERTAINTY_DIR.mkdir(parents=True, exist_ok=True)
missing_feature_results.to_csv(RUN_DIR / 'dp_missing_feature_metrics.csv', index=False)
uncertainty_changes.to_csv(RUN_DIR / 'uncertainty_vs_mean_only.csv', index=False)
uncertainty_auxiliary_results.to_csv(UNCERTAINTY_DIR / 'auxiliary_comparison.csv', index=False)
reconstruction_audit.to_csv(UNCERTAINTY_DIR / 'reconstruction_audit.csv', index=False)
public_roles = [dict(record_id=record_id, role=role)
    for role, ids in uncertainty_roles.items() for record_id in sorted(ids)]
pd.DataFrame(public_roles).to_csv(UNCERTAINTY_DIR / 'public_auxiliary_roles.csv', index=False)
auxiliary_attack_metrics.to_csv(RUN_DIR / 'auxiliary_attack_metrics.csv')
shadow_income_metrics.to_csv(RUN_DIR / 'shadow_income_metrics.csv')
for condition, history in training_histories.items():
    pd.DataFrame(history).to_csv(RUN_DIR / f'{condition}_public_validation_history.csv', index=False)
protocol = dict(study_variant='Main Exp 3 matched 100-epoch uncertainty-aware MF-MIA with DP targets',
    accounting_calibration='Exact private-loader steps; conservative Opacus accountant rate',
    execution_status='complete', target_year=TARGET_YEAR, auxiliary_year=AUXILIARY_YEAR,
    epsilon_budgets=EPSILON_BUDGETS, delta=DELTA, train_size=TRAIN_SIZE,
    feature_count=len(EXPANDED_FEATURES), features=EXPANDED_FEATURES,
    label='nominal PINCP > 50000', split_config=SPLIT_CONFIG, target_partition_sizes=partition_counts,
    public_preprocessing_records=PUBLIC_PREPROCESSING_SIZE,
    preprocessing_scope='Fixed public 2014 subset of full auxiliary pool; shared by every target; may overlap public shadows/donors',
    public_validation='Fixed public 2023 split; monitoring only, never selects/stops the 100-epoch targets',
    training_config=TRAIN_CONFIG, target_results=target_metadata,
    privacy_scope='Individual target model; record-level DP conditional on fixed partition/public data',
    privacy_bound='Fixed full 100-epoch schedule; final accountant epsilon must equal schedule bound',
    secure_mode=False, research_outputs_are_dp_releases=False,
    multiple_models='Per-model bounds; joint release requires composition; control is non-private',
    n_shadow_models=N_SHADOW_MODELS, shadow_algorithms=SHADOW_ALGORITHMS, shadow_seeds=shadow_seeds,
    shadow_algorithm_plan=SHADOW_PLAN, original_tree_shadow_count=N_TREE_SHADOWS,
    shadow_counts={algorithm: SHADOW_PLAN.count(algorithm) for algorithm in SHADOW_ALGORITHMS},
    shadow_mlp_config=SHADOW_MLP_CONFIG, shadow_mlp_training=shadow_mlp_training,
    records_per_class_per_shadow=SHADOW_RECORDS_PER_CLASS,
    attack='Frozen Exp 3 complete-feature RF transfer attack; 16 original trees plus 4 target-recipe MLPs',
    attack_features=list(attack_result['classifier'].feature_names_in_), include_blind_walk=True,
    walk_config=WALK_CONFIG, extra_numeric_protocol=EXPANDED_NUMERIC_WALK,
    walk_scope='Historical Exp 3 asymmetry retained: shadow commuting fields categorical; target numeric',
    shadow_extra_numeric_protocol=SHADOW_NUMERIC_WALK,
    exp3_reference_target_config=EXP3_REFERENCE_TARGET_CONFIG,
    exp3_comparison_exception='Public 2014 target preprocessing instead of private 2023 train-fitted preprocessing; use fresh non-private control',
    full_public_auxiliary_pool=True,
    shadow_query_audits=shadow_query_audits, auxiliary_attack_metrics=attack_result['metrics'],
    accuracy_threshold=attack_result['accuracy_threshold'], low_fpr_threshold=attack_result['low_fpr_threshold'],
    plots='Disabled; plot exported CSVs separately',
    versions=versions, checkpoints_enabled=True, checkpoint_signature=run_signature,
    checkpoint_directory=str(CHECKPOINTS.directory), graphs_enabled=False, logging_enabled=True, per_target_results_enabled=True,
    missing_feature_experiments=True, repeated_seed_confirmation=False,
    missing_feature_protocol=dict(k=UNCERTAINTY_K, masks=MISSING_MASKS,
        reconstruction_seed=RECONSTRUCTION_SEED, pfv_seed=SEED,
        reconstruction='2014-only conditional forest-leaf donors; no income or membership conditioning',
        reconstruction_chain_order=MASK_FIELDS, score_std_ddof=0,
        auxiliary_roles={name: len(ids) for name, ids in uncertainty_roles.items()},
        reconstruction_fit_records=len(recon_fit_ids), reconstruction_audit_records=len(recon_check_ids),
        public_preprocessing_records_excluded_from_donors=False,
        donors_exclude_all_attack_meta_calibration_test_records=True,
        methods=['pfv', 'single', 'average', 'learned_mean', 'uncertainty'],
        conditional_thresholds=frozen_thresholds,
        calibration='auxiliary only; frozen across all target budgets',
        auxiliary_completion_query_audits=uncertainty_query_audits,
        target_membership_labels_used_for_fitting=False,
        target_mia_records='unchanged Exp 4 balanced member/nonmember partitions',
        primary_contrast='uncertainty minus learned_mean on identical conditional draws',
        source_exp3='exp_3_expanded_missing_feature_baseline_mlp.ipynb',
        source_exp4='exp_4_differential_privacy_accuracy.ipynb'),
    missing_feature_plots='Disabled; per-mask metrics retained in CSV')
_ = (RUN_DIR / 'protocol.json').write_text(json.dumps(json_value(protocol), indent=2, allow_nan=False), encoding='utf-8')
print('Final metrics:', RUN_DIR / 'dp_target_and_mia_metrics.csv')


print('Missing-feature metrics:', RUN_DIR / 'dp_missing_feature_metrics.csv')
log_progress('Final metrics and protocol saved; plotting disabled')

Final metrics: /home/syedkhaj/MF-MIA-nebula/artifacts/dp-uncertainty/run-3734a3b6b6d89418/dp_target_and_mia_metrics.csv
Missing-feature metrics: /home/syedkhaj/MF-MIA-nebula/artifacts/dp-uncertainty/run-3734a3b6b6d89418/dp_missing_feature_metrics.csv


Implementation references:
[Opacus PrivacyEngine](https://opacus.ai/api/privacy_engine.html),
[BatchMemoryManager](https://opacus.ai/api/batch_memory_manager.html),
[DPDataLoader](https://opacus.ai/api/data_loader.html),
[Opacus FAQ](https://opacus.ai/docs/faq).